In [0]:
df=spark.read.format("csv").option("header","true").option("inferschema","true").load("abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/inbound/emp_Details.csv")
df.display()

In [0]:
display(df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

cust_schema=StructType([
  StructField("id",IntegerType(),True),
  StructField("name",StringType(),True),
  StructField("salary",IntegerType(),True),
  StructField("gender",StringType(),True),
  StructField("deptno",IntegerType(),True)
])
df=spark.read.format("csv").option("header",True).schema(cust_schema).load("abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/inbound/emp_Details.csv")
df.display()

### seperrated by |


In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").option("sep","|").load("abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/inbound/prsn_dtls.csv")

df.display()

### Bad records path

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

bad_schema=StructType([
  StructField("id",IntegerType(),True),
  StructField("name",StringType(),True),
  StructField("salary",IntegerType(),True),
  StructField("gender",StringType(),True),
  
])

In [0]:
badrecords_df=spark.read.format("csv").option("header","true").schema(bad_schema).option("badRecordsPath","abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/inbound/badrecords").load("abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/inbound/emp_badrecords.csv")

badrecords_df.display()

### Bronze layer external table

In [0]:
bronze_path = "abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/delta table/bronze_layer"
bronze_table = "databricksworkspace.default.bronze_emp_details"

# Read the raw employee CSV with the explicit schema defined earlier
bronze_df = (spark.read.format("csv")
             .option("header", "true")
             .schema(cust_schema)
             .load("abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/inbound/emp_Details.csv"))

# Write as Delta to the external location and register it as an external table
(bronze_df.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")  # needed to drop the previously written ingestion_ts column
 .option("path", bronze_path)
 .saveAsTable(bronze_table))

display(spark.sql(f"DESCRIBE DETAIL {bronze_table}").select("name", "location", "numFiles"))
display(spark.table(bronze_table))

In [0]:
%sql
ALTER TABLE databricksworkspace.default.bronze_emp_details
ALTER COLUMN salary SET TAGS ('PII' = 'personal information identification');

In [0]:
%sql
SELECT catalog_name, schema_name, table_name, column_name, tag_name, tag_value
FROM databricksworkspace.information_schema.column_tags
WHERE schema_name = 'default'
  AND table_name = 'bronze_emp_details';

### Silver layer external table
Cleansed version of the bronze table: trimmed and standardized text, null IDs removed, duplicates dropped.

In [0]:
from pyspark.sql.functions import col, trim, initcap, upper

silver_path = "abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/delta table/silver_layer"
silver_table = "databricksworkspace.default.silver_emp_details"

# Cleanse bronze data
silver_df = (spark.table("databricksworkspace.default.bronze_emp_details")
             .filter(col("id").isNotNull())
             .withColumn("name", initcap(trim(col("name"))))
             .withColumn("gender", upper(trim(col("gender"))))
             .dropDuplicates(["id"]))

# Write as Delta to the external location and register it as an external table
(silver_df.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .option("path", silver_path)
 .saveAsTable(silver_table))

display(spark.sql(f"DESCRIBE DETAIL {silver_table}").select("name", "location", "numFiles"))
display(spark.table(silver_table).orderBy("id"))

In [0]:
%sql
ALTER TABLE databricksworkspace.default.silver_emp_details
ALTER COLUMN salary SET TAGS ('PII' = 'personal information identification');

### Gold layer external table
Department-level salary summary aggregated from the silver table.

In [0]:
from pyspark.sql.functions import col, count, sum, avg, min, max, round, when

gold_path = "abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/delta table/gold_layer"
gold_table = "databricksworkspace.default.gold_dept_salary_summary"

# Aggregate silver data per department
gold_df = (spark.table("databricksworkspace.default.silver_emp_details")
           .groupBy("deptno")
           .agg(count("id").alias("employee_count"),
                sum(when(col("gender") == "M", 1).otherwise(0)).alias("male_count"),
                sum(when(col("gender") == "F", 1).otherwise(0)).alias("female_count"),
                sum("salary").alias("total_salary"),
                round(avg("salary"), 2).alias("avg_salary"),
                min("salary").alias("min_salary"),
                max("salary").alias("max_salary")))

# Write as Delta to the external location and register it as an external table
(gold_df.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .option("path", gold_path)
 .saveAsTable(gold_table))

display(spark.sql(f"DESCRIBE DETAIL {gold_table}").select("name", "location", "numFiles"))
display(spark.table(gold_table).orderBy("deptno"))

In [0]:
df=spark.read.format("csv").option("inferschema","true").option("header","true").load("abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/inbound/reviews.csv")
df.display()